# Handling Exceptions

## Table of Contents

1. [Basic Exception Handling](#1)
2. [Custom Exception Hierarchies](#2)
3. [When NOT to Catch Exceptions](#3)
4. [Nested Functions and Propagation](#4)
5. [contextlib.suppress](#5)
6. [Exercise: Harden the Pipeline](#6)

---
## 1. Basic Exception Handling <a id="1"></a>

Exception handling keeps your pipeline running gracefully when things go wrong — while still surfacing meaningful error information.

In [ ]:
import pandas as pd
import numpy as np


def load_dataset(path: str) -> pd.DataFrame:
    """Load a CSV with descriptive errors.

    Args:
        path: Path to the CSV file.

    Returns:
        Loaded DataFrame.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file is empty.
    """
    try:
        df = pd.read_csv(path)
        if df.empty:
            raise ValueError(f"File exists but is empty: {path}")
        return df
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Dataset not found: {path}\n"
            f"Check the path and ensure the data pipeline has run first."
        )
    except pd.errors.EmptyDataError:
        raise ValueError(f"File is completely empty (0 bytes): {path}")
    # Do NOT catch generic Exception here — let unexpected errors propagate


# Test each case
try:
    load_dataset("nonexistent.csv")
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")

---
## 2. Custom Exception Hierarchies <a id="2"></a>

Custom exceptions let callers handle specific error types without inspecting error messages. They also document what can go wrong — which is documentation your tests can enforce.

In [ ]:
# Define a hierarchy rooted at a project-level base exception
class PipelineError(Exception):
    """Base class for all pipeline errors."""


class DataError(PipelineError):
    """A problem with the input data."""


class SchemaError(DataError):
    """Required columns are missing or have the wrong type."""
    def __init__(self, missing_columns: list[str]):
        self.missing_columns = missing_columns
        super().__init__(f"Missing required columns: {missing_columns}")


class MissingValuesError(DataError):
    """Data contains more missing values than the allowed threshold."""
    def __init__(self, column: str, pct_missing: float, threshold: float):
        self.column = column
        self.pct_missing = pct_missing
        self.threshold = threshold
        super().__init__(
            f"Column '{column}' is {pct_missing:.1%} missing "
            f"(threshold: {threshold:.1%})"
        )


class ModelError(PipelineError):
    """A problem during model training or inference."""


class InsufficientDataError(ModelError):
    """Not enough samples to train reliably."""
    def __init__(self, n_samples: int, min_required: int):
        super().__init__(
            f"Need at least {min_required} samples, got {n_samples}"
        )


# Demonstrate the hierarchy
def validate_schema(df: pd.DataFrame, required: list[str]) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise SchemaError(missing)

def check_missing_values(df: pd.DataFrame, column: str, threshold: float = 0.05) -> None:
    pct = df[column].isnull().mean()
    if pct > threshold:
        raise MissingValuesError(column, pct, threshold)


# Test — catch the base class to handle all data problems uniformly
df_bad = pd.DataFrame({"tenure": [1, None, None, 4, None], "wrong_name": [10, 20, 30, 40, 50]})

try:
    validate_schema(df_bad, ["tenure", "MonthlyCharges"])
except SchemaError as e:
    print(f"SchemaError: {e}")
    print(f"Missing columns list: {e.missing_columns}")

try:
    check_missing_values(df_bad, "tenure", threshold=0.10)
except MissingValuesError as e:
    print(f"MissingValuesError: {e}")
    print(f"Column: {e.column}, Pct missing: {e.pct_missing:.1%}")

---
## 3. When NOT to Catch Exceptions <a id="3"></a>

A common mistake is catching too broadly. Swallowing errors silently is worse than crashing — you corrupt results without knowing it.

In [ ]:
# BAD: bare except swallows EVERYTHING, including bugs in your own code
def bad_preprocessing(df):
    try:
        result = df["nonexistent_column"] * 2   # a real bug — typo in column name
    except:   # noqa
        pass  # silently continues with broken data
    return df

# BAD: catching Exception is almost as bad
def also_bad(df):
    try:
        result = df["nonexistent_column"] * 2
    except Exception:
        pass  # still swallows KeyError, TypeError, MemoryError, etc.
    return df

# GOOD: catch only what you know how to handle
def good_preprocessing(df: pd.DataFrame, column: str, default: float = 0.0) -> pd.DataFrame:
    df = df.copy()
    try:
        df[column] = df[column] * 2
    except KeyError:
        # We specifically handle: column not found. We add it with a default.
        import logging
        logging.getLogger(__name__).warning(
            f"Column '{column}' not found — adding with default {default}"
        )
        df[column] = default
    # All other exceptions (MemoryError, TypeError, etc.) propagate naturally
    return df


df_test = pd.DataFrame({"tenure": [1, 2, 3]})
result = good_preprocessing(df_test, "missing_col", default=99.0)
print(result)

**Rule of thumb**: only catch an exception if you can *do something useful* with it — log it with context, retry, fall back to a default, or raise a better error. If you can't do any of those, let it propagate.

---
## 4. Nested Functions and Propagation <a id="4"></a>

In a pipeline with multiple nested steps, exceptions should propagate up and be handled **once**, at the right level — not caught and re-raised redundantly at every level.

In [ ]:
import logging

log = logging.getLogger("pipeline")


class ChurnPipeline:
    """A multi-step pipeline that demonstrates clean error propagation."""

    REQUIRED_COLS = ["tenure", "MonthlyCharges", "Churn"]
    MAX_MISSING_PCT = 0.10

    def run(self, df: pd.DataFrame) -> dict:
        """Run the full pipeline. Raises PipelineError on data/model issues.

        The outer method is the only place that decides what to do with errors.
        Inner methods just raise cleanly and propagate.
        """
        try:
            log.info(f"Pipeline started | {len(df)} rows")
            df = self._validate(df)
            df = self._clean(df)
            model, acc = self._train(df)
            log.info(f"Pipeline complete | accuracy={acc:.3f}")
            return {"model": model, "accuracy": acc}

        except SchemaError as e:
            log.error(f"Schema validation failed: {e}")
            raise  # re-raise — the caller decides what to do
        except DataError as e:
            log.error(f"Data quality issue: {e}")
            raise
        except ModelError as e:
            log.error(f"Model error: {e}")
            raise
        # We do NOT catch the generic PipelineError base — unexpected subtypes should surface

    def _validate(self, df: pd.DataFrame) -> pd.DataFrame:
        missing = [c for c in self.REQUIRED_COLS if c not in df.columns]
        if missing:
            raise SchemaError(missing)  # just raise — no logging here
        return df

    def _clean(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in ["tenure", "MonthlyCharges"]:
            pct = df[col].isnull().mean()
            if pct > self.MAX_MISSING_PCT:
                raise MissingValuesError(col, pct, self.MAX_MISSING_PCT)
        return df.dropna(subset=["tenure", "MonthlyCharges"])

    def _train(self, df: pd.DataFrame) -> tuple:
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import accuracy_score
        from sklearn.preprocessing import LabelEncoder

        X = df[["tenure", "MonthlyCharges"]]
        y = LabelEncoder().fit_transform(df["Churn"].astype(str))

        if len(df) < 50:
            raise InsufficientDataError(len(df), 50)

        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
        clf = RandomForestClassifier(n_estimators=50, random_state=42)
        clf.fit(X_tr, y_tr)
        return clf, accuracy_score(y_te, clf.predict(X_te))


# Happy path
np.random.seed(42)
good_df = pd.DataFrame({
    "tenure": np.random.randint(1, 60, 200),
    "MonthlyCharges": np.random.uniform(20, 100, 200),
    "Churn": np.random.choice(["Yes", "No"], 200),
})

pipeline = ChurnPipeline()
result = pipeline.run(good_df)
print(f"Accuracy: {result['accuracy']:.3f}")

print()

# Schema error path
bad_df = good_df.drop(columns=["Churn"])
try:
    pipeline.run(bad_df)
except SchemaError as e:
    print(f"Caught SchemaError: {e}")

---
## 5. contextlib.suppress <a id="5"></a>

For the rare case where you genuinely want to swallow a specific exception, `contextlib.suppress` is cleaner than `try/except: pass` — and it's explicit about your intent.

In [ ]:
from contextlib import suppress
import os

# Instead of:
try:
    os.remove("/tmp/nonexistent_file.txt")
except FileNotFoundError:
    pass

# Use:
with suppress(FileNotFoundError):
    os.remove("/tmp/nonexistent_file.txt")

print("No error — suppress worked as expected")

# ML use case: ignore convergence warnings during a quick test
from sklearn.exceptions import ConvergenceWarning
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    from sklearn.linear_model import LogisticRegression
    lr = LogisticRegression(max_iter=1).fit(
        np.random.randn(50, 2), np.random.randint(0, 2, 50)
    )
print("Logistic regression fitted (convergence warning suppressed)")

---
## 6. Exercise: Harden the Pipeline <a id="6"></a>

The function below is brittle. Add proper exception handling:

1. Define `FeatureEngineeringError(PipelineError)` and `ModelTrainingError(PipelineError)`
2. Wrap the feature engineering step — raise `FeatureEngineeringError` if any feature column is missing
3. Wrap the model training step — raise `ModelTrainingError` if training fails for any reason
4. At the top level, catch each type, log it appropriately, and return `None` instead of crashing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Your exception classes here


def run_experiment(df: pd.DataFrame, feature_cols: list, target_col: str) -> float | None:
    """Run an ML experiment. Returns accuracy or None on failure.
    
    Add proper exception handling.
    """
    # Feature engineering
    X = df[feature_cols]  # Will KeyError if a column is missing
    y = df[target_col]

    # Model training
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    return acc


# Test cases
np.random.seed(42)
df = pd.DataFrame({
    "tenure": np.random.randint(1, 60, 200),
    "charges": np.random.uniform(20, 100, 200),
    "churn": np.random.randint(0, 2, 200),
})

# Good run
result = run_experiment(df, ["tenure", "charges"], "churn")
print(f"Good run: {result}")

# Bad run — wrong feature name
result = run_experiment(df, ["tenure", "wrong_col"], "churn")
print(f"Bad run: {result}")  # should return None, not crash